In [110]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline

In [111]:
data_train=pd.read_csv('Titanic_train.csv')
data_test=pd.read_csv('Titanic_test.csv')
#data_train.head().T

In [112]:
data_train.describe(include='all')

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


In [113]:
data_train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [114]:
#sex映射
data_train['Sex'] = data_train['Sex'].map({'male': 0, 'female': 1})
data_test['Sex'] = data_test['Sex'].map({'male': 0, 'female': 1})

In [115]:
#合成新特征familysize，is_alone（用S列和P列）
data_train['FamilySize']=data_train['SibSp']+data_train['Parch']
data_test['FamilySize']=data_test['SibSp']+data_test['Parch']
data_train['is_alone']=np.where(data_train['FamilySize']==0,1,0)
data_test['is_alone']=np.where(data_test['FamilySize']==0,1,0)

In [116]:
#姓氏头衔和年龄有关，通过一种头衔年龄平均数确定缺失值补充
data_train['Title'] = data_train['Name'].str.extract(r' ([A-Za-z]+)\.')
#合并稀有称呼
title_counts=data_train['Title'].value_counts()
rare=title_counts[title_counts<10].index
data_train['Title']=data_train['Title'].replace(rare,'Rare')
print(data_train['Title'].value_counts())
#每一种头衔的平均年龄，以此来填充对应缺失值的age
data_train['Age']=data_train.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
print(data_train['Age'].isnull().sum())

#姓氏头衔和年龄有关，通过一种头衔年龄平均数确定缺失值补充
data_test['Title'] = data_test['Name'].str.extract(r' ([A-Za-z]+)\.')
#合并稀有称呼
title_counts=data_test['Title'].value_counts()
rare=title_counts[title_counts<10].index
data_test['Title']=data_test['Title'].replace(rare,'Rare')
print(data_test['Title'].value_counts())
#每一种头衔的平均年龄，以此来填充对应缺失值的age
data_test['Age']=data_test.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
print(data_test['Age'].isnull().sum())

#todo把年龄的补充值变成中位数，还有我想一开始就把age去量纲化

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64
0
Title
Mr        240
Miss       78
Mrs        72
Master     21
Rare        7
Name: count, dtype: int64
0


In [117]:
data_test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
FamilySize       0
is_alone         0
Title            0
dtype: int64

In [118]:
#Cabin缺失值太多了,直接用是否有船舱号标码
data_train['HasCabin']=data_train['Cabin'].notnull().astype(int)
data_test['HasCabin']=data_test['Cabin'].notnull().astype(int)

In [119]:
#Fare进行区间离散映射
data_train['FareBin']=pd.cut(data_train['Fare'],bins=[-1,7.91,14.45,31,1000],labels=[0,1,2,3]).astype(int)
data_test['Fare']=data_test['Fare'].fillna(data_test['Fare'].median())#这个地方的补充值也变成median
data_test['FareBin']=pd.cut(data_test['Fare'],bins=[-1,7.91,14.45,31,1000],labels=[0,1,2,3]).astype(int)

In [120]:
#embarked独热编码   todo这里embark和title做独热编码要避免多重线性
data_train['Embarked']=data_train['Embarked'].fillna(data_train['Embarked'].mode())
data_train = pd.get_dummies(data_train, columns=['Embarked'], prefix='Embarked',drop_first=True,dtype=int)
data_test['Embarked']=data_test['Embarked'].fillna(data_test['Embarked'].mode())
data_test= pd.get_dummies(data_test, columns=['Embarked'], prefix='Embarked',drop_first=True,dtype=int)

In [121]:
data_train['ageclass']=data_train['Age']*data_train['Pclass']
data_train['agefare']=data_train['Age']*data_train['Fare']
data_test['ageclass']=data_test['Age']*data_test['Pclass']
data_test['agefare']=data_test['Age']*data_test['Fare']

In [122]:
#先删除明显没有用的特征passengerid
data_train=data_train.drop(columns=['PassengerId'])
data_test=data_test.drop(columns=['PassengerId'])
#删除原来的改造过的特征
data_train=data_train.drop(columns=['Name'])
data_train=data_train.drop(columns=['Cabin'])
data_train=data_train.drop(columns=['Ticket'])
data_train=data_train.drop(columns=['Fare'])
#删除原来的改造过的特征
data_test=data_test.drop(columns=['Name'])
data_test=data_test.drop(columns=['Cabin'])
data_test=data_test.drop(columns=['Ticket'])
data_test=data_test.drop(columns=['Fare'])

In [123]:
#Title做独热编码
data_train = pd.get_dummies(data_train, columns=['Title'], prefix='Title',drop_first=True,dtype=int)
data_test = pd.get_dummies(data_test, columns=['Title'], prefix='Title',drop_first=True,dtype=int)

In [124]:
#pclass进行独热编码
data_train = pd.get_dummies(data_train, columns=['Pclass'], prefix='Pclass',drop_first=True,dtype=int)
data_test = pd.get_dummies(data_test, columns=['Pclass'], prefix='Pclass',drop_first=True,dtype=int)

In [125]:
data_train.head()

,Survived,Sex,Age,SibSp,Parch,FamilySize,is_alone,HasCabin,FareBin,Embarked_Q,Embarked_S,ageclass,agefare,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Pclass_2,Pclass_3
0,0,0,22.0,1,0,1,0,0,0,0,1,66.0,159.5000,0,1,0,0,0,1
1,1,1,38.0,1,0,1,0,1,3,0,0,38.0,2708.7654,0,0,1,0,0,0
2,1,1,26.0,0,0,0,1,0,1,0,1,78.0,206.0500,1,0,0,0,0,1
3,1,1,35.0,1,0,1,0,1,3,0,1,35.0,1858.5000,0,0,1,0,0,0
4,0,0,35.0,0,0,0,1,0,1,0,1,105.0,281.7500,0,1,0,0,0,1


In [126]:
data_train.describe(include='all')

,Survived,Sex,Age,SibSp,Parch,FamilySize,is_alone,HasCabin,FareBin,Embarked_Q,Embarked_S,ageclass,agefare,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Pclass_2,Pclass_3
count,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000
mean,0.383838,0.352413,29.387957,0.523008,0.381594,0.904602,0.602694,0.228956,1.505051,0.086420,0.722783,63.927430,1010.891616,0.204265,0.580247,0.140292,0.030303,0.206510,0.551066
std,0.486592,0.477990,13.262592,1.102743,0.806057,1.613459,0.489615,0.420397,1.118148,0.281141,0.447876,32.952241,1808.903717,0.403390,0.493796,0.347485,0.171516,0.405028,0.497665
min,0.000000,0.000000,0.420000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.920000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,21.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,40.000000,206.831250,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,30.000000,0.000000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000,1.000000,62.000000,354.200000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,1.000000,35.000000,1.000000,0.000000,1.000000,1.000000,0.000000,2.000000,0.000000,1.000000,90.000000,1011.450000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,80.000000,8.000000,6.000000,10.000000,1.000000,1.000000,3.000000,1.000000,1.000000,222.000000,18443.851200,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [127]:
data_test.head()

,Sex,Age,SibSp,Parch,FamilySize,is_alone,HasCabin,FareBin,Embarked_Q,Embarked_S,ageclass,agefare,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Pclass_2,Pclass_3
0,0,34.5,0,0,0,1,0,0,1,0,103.5,270.1074,0,1,0,0,0,1
1,1,47.0,1,0,1,0,0,0,0,1,141.0,329.0000,0,0,1,0,0,1
2,0,62.0,0,0,0,1,0,1,1,0,124.0,600.6250,0,1,0,0,1,0
3,0,27.0,0,0,0,1,0,1,0,1,81.0,233.8875,0,1,0,0,0,1
4,1,22.0,1,1,2,0,0,1,0,1,66.0,270.3250,0,0,1,0,0,1


In [128]:
data_test.describe(include='all')

,Sex,Age,SibSp,Parch,FamilySize,is_alone,HasCabin,FareBin,Embarked_Q,Embarked_S,ageclass,agefare,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Pclass_2,Pclass_3
count,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000
mean,0.363636,29.712919,0.447368,0.392344,0.839713,0.605263,0.217703,1.490431,0.110048,0.645933,62.229856,1298.477339,0.186603,0.574163,0.172249,0.016746,0.222488,0.521531
std,0.481622,12.985445,0.896760,0.981429,1.519072,0.489380,0.413179,1.146846,0.313324,0.478803,28.748095,2713.217363,0.390059,0.495062,0.378049,0.128474,0.416416,0.500135
min,0.000000,0.170000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.510000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,22.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,42.000000,206.032200,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,28.500000,0.000000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000,1.000000,60.000000,346.500000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,36.500000,1.000000,0.000000,1.000000,1.000000,0.000000,3.000000,0.000000,1.000000,85.500000,1115.100000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,76.000000,8.000000,9.000000,10.000000,1.000000,1.000000,3.000000,1.000000,1.000000,181.500000,29715.093600,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold,GridSearchCV,RandomizedSearchCV

In [130]:
data_train_label=data_train['Survived']
data_train_feature=data_train.drop(columns=['Survived'])
data_test_feature=data_test

In [131]:
kf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
# 1. 定义你要比较的模型
models = {
    "逻辑回归": LogisticRegression(max_iter=1000),
    "随机森林": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42,verbose=-1)
}

# 2. 循环跑交叉验证，收集数据
results_list = []
for name, model in models.items():
    cv_results = cross_validate(
        model, data_train_feature, data_train_label,
        cv=kf,
        scoring='accuracy',
        return_train_score=True
    )
    train_mean = cv_results['train_score'].mean()
    val_mean = cv_results['test_score'].mean()
    gap = train_mean - val_mean
    
    results_list.append({
        "模型": name,
        "训练分": round(train_mean, 4),
        "验证分": round(val_mean, 4),
        "差值(过拟合程度)": round(gap, 4)
    })

# 3. 打印对比表格
df_results = pd.DataFrame(results_list)
print(df_results.sort_values(by="验证分", ascending=False))

         模型     训练分     验证分  差值(过拟合程度)
3  LightGBM  0.9214  0.8361     0.0853
0      逻辑回归  0.8364  0.8271     0.0093
1      随机森林  0.9857  0.8148     0.1709
2       SVM  0.6661  0.6644     0.0017


In [ ]:
# base_model=LGBMClassifier(random_state=42,verbose=-1)
# base_cv=cross_validate(base_model,data_train_feature,data_train_label,cv=kf,scoring='accuracy',return_train_score=True)
# base_train = base_cv['train_score'].mean()
# base_val = base_cv['test_score'].mean()
# base_gap = base_train - base_val

# print(f"【调参前】训练分: {base_train:.4f}, 验证分: {base_val:.4f}, 差值: {base_gap:.4f}")

【调参前】训练分: 0.9669, 验证分: 0.8204, 差值: 0.1465


In [ ]:
# param={
#     'num_leaves':[20,31,40,50],
#     'max_depth':[4,5,6,7],
#     'min_data_in_leaf':[10,20,30],
#     'min_gain_to_split':[0.5,1,1.5],
#     'reg_lambda':[0,1,5,10],
#     'n_estimators':[100,200,300,500],
#     'learning_rate':[0.01,0.05,0.1]
#     }
# random_search=RandomizedSearchCV(LGBMClassifier(random_state=42),param_distributions=param,cv=kf,scoring='accuracy',n_jobs=-1,n_iter=100,random_state=42,verbose=0)
# random_search.fit(data_train_feature,data_train_label);

In [ ]:
# print(random_search.best_params_)

{'reg_lambda': 1, 'num_leaves': 31, 'n_estimators': 300, 'min_gain_to_split': 0.5, 'min_data_in_leaf': 20, 'max_depth': 4, 'learning_rate': 0.1}


In [ ]:
# # ========== 第3步：调参后的评估 ==========
# best_model = random_search.best_estimator_

# after_cv = cross_validate(
#     best_model, data_train_feature, data_train_label,
#     cv=kf, scoring='accuracy', return_train_score=True,verbose=0
# )
# after_train = after_cv['train_score'].mean()
# after_val = after_cv['test_score'].mean()
# after_gap = after_train - after_val

# print(f"【调参后】训练分: {after_train:.4f}, 验证分: {after_val:.4f}, 差值: {after_gap:.4f}")

【调参后】训练分: 0.8808, 验证分: 0.8417, 差值: 0.0390


In [137]:
#逻辑回归
base_model2=LogisticRegression(random_state=42)
base_cv=cross_validate(base_model2,data_train_feature,data_train_label,cv=kf,scoring='accuracy',return_train_score=True)
base_train = base_cv['train_score'].mean()
base_val = base_cv['test_score'].mean()
base_gap = base_train - base_val

print(f"【调参前】训练分: {base_train:.4f}, 验证分: {base_val:.4f}, 差值: {base_gap:.4f}")

【调参前】训练分: 0.8215, 验证分: 0.8160, 差值: 0.0056


In [138]:
param2={
     'logisticregression__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'logisticregression__penalty': ['l1', 'l2'],
    'logisticregression__solver': ['liblinear', 'saga'],
    'logisticregression__max_iter': [1000, 2000]                
}
lr_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42)
)
random_search2=RandomizedSearchCV(lr_pipe,param_distributions=param2,cv=kf,scoring='accuracy',n_jobs=-1,n_iter=100,random_state=42)
random_search2.fit(data_train_feature,data_train_label);

In [139]:
print(random_search2.best_params_)

{'logisticregression__solver': 'saga', 'logisticregression__penalty': 'l2', 'logisticregression__max_iter': 1000, 'logisticregression__C': 0.1}


In [140]:
# ========== 第3步：调参后的评估 ==========
best_model2 = random_search2.best_estimator_

after_cv = cross_validate(
    best_model2, data_train_feature, data_train_label,
    cv=kf, scoring='accuracy', return_train_score=True
)
after_train = after_cv['train_score'].mean()
after_val = after_cv['test_score'].mean()
after_gap = after_train - after_val

print(f"【调参后】训练分: {after_train:.4f}, 验证分: {after_val:.4f}, 差值: {after_gap:.4f}")

【调参后】训练分: 0.8364, 验证分: 0.8328, 差值: 0.0037


In [ ]:
best_model2.fit(data_train_feature, data_train_label)
data_test_label_pred = best_model2.predict(data_test_feature)
df=pd.read_csv('Titanic_test.csv')
df['Survived']=data_test_label_pred
df.to_csv('Titanic_submission4.csv')

In [ ]:
# from sklearn.ensemble import VotingClassifier
# ensemble = VotingClassifier(
#     estimators=[('lr', best_model2), ('lgb', best_model)],
#     voting='soft',  # 软投票
#     weights=[1,3]
# )
# ensemble_scores = cross_validate(ensemble, data_train_feature, data_train_label, cv=kf, scoring='accuracy',return_train_score=True)
# print(f"【调参后】训练分: {ensemble_scores['train_score'].mean():.4f}, 验证分: {ensemble_scores['test_score'].mean():.4f}")

【调参后】训练分: 0.8749, 验证分: 0.8373


In [ ]:
# ensemble.fit(data_train_feature, data_train_label)
# data_test_label_pred = ensemble.predict(data_test_feature)
# df=pd.read_csv('Titanic_test.csv')
# df['Survived']=data_test_label_pred
# df.to_csv('Titanic_submission5.csv')

最终写入文件